# Wordle solver

The solver itself now lives in modules next to this notebook, so this notebook
keeps only the analysis that motivates the priors:

| module | contents |
| --- | --- |
| `wordle_data.py` | word lists, the feedback table, past solutions |
| `priors.py` | the two toggleable priors -> a weight per word |
| `kernels.py` | numba scorers: information gain, E(g, S), and the two-turn value |
| `estimator.py` | the forms of f(n), their least-squares fit, and the parameter cache |
| `exact.py` | exact endgame search, used wherever it is affordable |
| `solver.py` | picking a guess, applying feedback, playing a game out |
| `train.py` | policy iteration, and self-play evaluation |
| `wordle_solver.py` | command line front end (`play` / `train` / `benchmark` / `compare` / `info`) |
| `wordle_game.py` | play Wordle yourself, against a chosen or a sampled answer |


In [ ]:
import json
import numpy as np

import estimator
import train
import wordle_data
from priors import ALL_CONFIGS, PriorConfig, build_weights
from solver import ENTROPY, EXPECTED, WordleSolver
from wordle_data import feedback_to_num, load_data

In [ ]:
"""start_date = date(2021, 6, 19)
end_date = date.today()
delta = end_date - start_date
all_dates = [(start_date + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(delta.days + 1)]

solutions_so_far = []
for date in all_dates:
    url = f"https://www.nytimes.com/svc/wordle/v2/{date}.json"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        solutions_so_far.append(data)
    else:
        print("Failed to retrieve today's Wordle data.")

with open("solutions.json", "w") as file:
    json.dump(solutions_so_far, file, indent=4)"""

## Data

In [ ]:
mask_last = 1  # today's answer is not known yet
data = load_data(mask_last=mask_last)

all_words = data.all_words
curated_list = data.curated_list
all_sols = data.past_solutions  # answers used so far, in order

len(all_words), len(curated_list), len(all_sols)

## How often is the answer off the curated list?

Tracy Bennett starts editing at index 506.

In [ ]:
start_N = 1830
nc_sols = 0
for sol in all_sols[start_N:]:
    if sol not in curated_list:
        nc_sols += 1
total_sols = len(all_sols[start_N:])
print(nc_sols, total_sols)
print(nc_sols / total_sols)

## How often is the answer a repeat?

In [ ]:
# 1689 was the first repeat word
start_N = 1689
repeat_sols = 0
for i in range(start_N, len(all_sols)):
    if all_sols[i] in all_sols[:i]:
        repeat_sols += 1
total_sols = len(all_sols) - start_N
print(repeat_sols, total_sols)
print(repeat_sols / total_sols)

## The priors

Both estimates land near 0.10, which is where `priors.py` gets its defaults.
Each prior can be switched off on its own, giving four weightings:

In [ ]:
for config in ALL_CONFIGS:
    weights = build_weights(data, config)
    print(f"{config.slug}: {config.describe()}")
    print(f"    distinct weights {len(np.unique(weights))}, "
          f"min {weights.min():.3e}, max {weights.max():.3e}, "
          f"effective vocabulary {estimator.effective_size(weights):.0f}")

## The estimator

`f(n)` estimates how many guesses are still needed from a position whose
candidate set has effective size `n = 2 ** entropy`, and a guess is scored by

$$E(g, S) = 1 + \sum_{r \neq \text{GGGGG}} p_r \, f(n_r)$$

Excluding the winning pattern is what makes a guess that could itself be the
answer worth more than an equally informative guess that could not.

Parameters come from policy iteration and are cached per prior combination in
`estimator_cache/`.  Re-fit everything with `python wordle_solver.py train`, or
one combination here:

In [ ]:
for config in ALL_CONFIGS:
    record = estimator.load_record(config)
    if record is None:
        print(f"{config.slug}: not fitted yet")
        continue
    form = estimator.get_form(record["form"])
    print(f"{config.slug}: opening {record['opening_word']}, "
          f"mean {record['best_train_mean_guesses']:.4f} guesses "
          f"(entropy baseline {record['entropy_baseline_mean_guesses']:.4f})")
    print("    " + estimator.describe(record["params"], form).replace("\n", "\n    "))

In [ ]:
# params, meta = train.policy_iteration(PriorConfig(), data=data, n_games=2000)

### Forms of f

Any basis that is linear in its parameters can be used, so each fit stays a
single least-squares solve.  `wordle_solver.py compare` runs policy iteration
once per form and tabulates the result; the shape that matters is how concave
the curve is in `x = log2(n)`.

With `f` exactly linear in x, minimizing E is equivalent to maximizing
`b * (information gain) + a * P(guess is the answer)` -- greedy entropy plus a
bonus for possible answers -- which is why `loglinear` picks the same opening
word as entropy mode and gains the least.

In [ ]:
for name, form in sorted(estimator.FORMS.items()):
    record = estimator.load_record(PriorConfig(), form)
    fitted = "" if record is None else f"  ->  opening {record['opening_word']}"
    marker = " (default)" if name == estimator.DEFAULT_FORM else ""
    print(f"{name:<12} {form.description:<40}{fitted}{marker}")

In [ ]:
# rows = train.compare_forms(PriorConfig(), data=data, n_games=2000, holdout_games=5000)

## Greedy entropy vs. minimizing expected guesses

In [ ]:
config = PriorConfig()  # both priors on
for mode in (ENTROPY, EXPECTED):
    solver = WordleSolver(data=data, config=config, mode=mode)
    train.evaluate(solver, n_games=300)

## The endgame is solved exactly

`f` is only needed where the game is too wide to search.  Once the candidate
set is small, or few turns remain, `exact.py` computes

$$V(S, t) = \min_g \left[ 1 + \sum_{r \neq \text{GGGGG}} p_r \, V(S_r, t-1) \right],
\qquad V(S, 0) = \text{loss penalty}$$

outright.  A loss penalty of 1 means a lost game scores 7, so the deadline
needs no separate objective, and the last-turn rule falls out of `V(S, 1)`.
Two turns or fewer is a closed form over all guesses, no cheaper than scoring
them heuristically, so it is always used.

Note the value *rises* with the number of turns left: with one turn to go,
being wrong costs only the penalty, while with more turns you are obliged to
keep guessing until you finish.

In [ ]:
# play two turns against a known answer, so the position is a real one
solver = WordleSolver(data=data, config=PriorConfig())
answer = data.word2ind["prowl"]
candidates = solver.all_candidates()
for turn in (1, 2):
    guess, _ = solver.best_guess(candidates, guesses_left=7 - turn)
    print(f"guess {turn}: {all_words[guess]}")
    candidates = solver.filter(guess, solver.feedback(guess, answer), candidates)

print(f"\n{len(candidates)} candidates left: "
      f"{', '.join(all_words[i] for i in candidates[:12])}")
for turns in (1, 2, 3, 6):
    guess, value = solver.exact.solve(candidates, turns)
    print(f"  {turns} turn(s) left: play {all_words[guess]}, "
          f"expected score {value:.4f}")

## Solving a real puzzle

`python wordle_solver.py play` does this from a terminal; the same thing
inline:

In [ ]:
def main(config=PriorConfig(), mode=EXPECTED, max_guesses=6):
    solver = WordleSolver(data=data, config=config, mode=mode)
    candidates = solver.all_candidates()
    for turn in range(1, max_guesses + 1):
        guess, score = solver.best_guess(candidates, guesses_left=max_guesses - turn + 1)
        unit = "bits" if mode == ENTROPY else "expected guesses"
        print(f"Recommended word: {all_words[guess]}   ({score:.4f} {unit}, "
              f"{len(candidates)} candidates left)")
        if len(candidates) == 1:
            print("This is the only possible answer.")

        played = input("Enter the word you guessed: ").strip().lower() or all_words[guess]
        feedback = input("Enter the 5 feedback tiles (G green, Y yellow, N gray): ")
        if feedback.strip().upper() == "GGGGG":
            print(f"You solved the wordle in {turn} guesses.")
            return turn
        candidates = solver.filter(
            data.word2ind[played], feedback_to_num(feedback), candidates
        )
        if len(candidates) == 0:
            print("No words match that feedback.")
            return None
    print("Out of guesses.")
    return None

In [ ]:
main()